## Dataset analysis: `student_grades_list15.csv`

Purpose: course-level outcomes (List15). Useful for deeper diagnostics (fail types FCW/FEX/MEX) and performance signals.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
DATA_DIR = Path.cwd()
df = pd.read_csv(DATA_DIR / "student_grades_list15.csv")
df.shape

In [ ]:
df.head()

In [ ]:
key = ["REG_NO", "SEMESTER_INDEX", "COURSE_CODE"]
df.duplicated(key).sum(), df[key].isna().any(axis=1).sum()

In [ ]:
for c in ["CW_MARK_60","EXAM_MARK_40","FINAL_MARK_100","COURSE_UNITS","GRADE_POINTS"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df[["CW_MARK_60","EXAM_MARK_40","FINAL_MARK_100","COURSE_UNITS","GRADE_POINTS"]].describe().T

In [ ]:
df["STATUS"].astype(str).str.upper().str.strip().value_counts(dropna=False)

In [ ]:
df["LETTER_GRADE"].value_counts(dropna=False)

## Advanced analytics

Focus: failure patterns, grade distributions, and their relationship to CGPA (via transcript join).

In [ ]:
from analysis_utils import basic_profile, missingness_report, numeric_outlier_report, merge_to_transcript_for_cgpa

print(basic_profile(df))
missingness_report(df, top_n=20)

In [ ]:
# Outlier and failure-pattern analysis + CGPA relationship
for c in ["FINAL_MARK_100","GRADE_POINTS"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

print(numeric_outlier_report(df, cols=["FINAL_MARK_100","GRADE_POINTS"]).head(10))

trans = pd.read_csv(DATA_DIR / "student_transcript_list15.csv")
trans["CGPA"] = pd.to_numeric(trans["CGPA"], errors="coerce")

# Derive per-semester failure intensity
x = df.copy()
x["STATUS_CLEAN"] = x["STATUS"].astype(str).str.upper().str.strip()
x["is_fail"] = x["STATUS_CLEAN"].isin(["F","FCW","FEX","MEX"]).astype(int)

agg = x.groupby(["REG_NO","SEMESTER_INDEX"], as_index=False).agg(
    courses=("COURSE_CODE","count"),
    fail_rate=("is_fail","mean"),
    mex_rate=("STATUS_CLEAN", lambda s: float((s=="MEX").mean())),
)

joined = merge_to_transcript_for_cgpa(agg, trans, on=["REG_NO","SEMESTER_INDEX"], how="inner")
joined[["CGPA","fail_rate","mex_rate","courses"]].corr(numeric_only=True)